In [ ]:
import numpy as np
import anndata
import dynamo as dyn
import matplotlib.pyplot as plt


# -------------------------------------------------
# CHOOSE DATASET HERE
# -------------------------------------------------


# -----------------------------
# DATASET REGISTRY
# -----------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
        "embedding": "./data/real_data_benchmark/cell_cycle/flowmap_embedding.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/pancreas/scvelo_umap_embedding.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_dynamical.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/dentate_gyrus/scvelo_umap_embedding.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/distance_pseudotime.npy",
        "embedding": "./data/real_data_benchmark/larry/flowmap_embedding.npy",
    },
}

In [ ]:
dataset_name = "cell_cycle"
# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
paths = DATASETS[dataset_name]

X = np.load(paths["X"])
V = np.load(paths["V"])
C = np.load(paths["color"])
X_emb = np.load(paths["embedding"])   # <-- use provided embedding

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}, V shape: {V.shape}, embedding: {X_emb.shape}")

# -------------------------------------------------
# BUILD ANNDATA
# -------------------------------------------------
adata = anndata.AnnData(X)
adata.var_names = [f"dim{j}" for j in range(X.shape[1])]
adata.obs["color"] = C
adata.layers["X_raw"] = X
adata.layers["V_raw"] = V
adata.obsm["X_umap"] = X_emb

# -------------------------------------------------
# FIX: Build neighborhood graph (required by Dynamo)
# -------------------------------------------------
dyn.tl.neighbors(
    adata,
    X_data=X,          # <--- tell Dynamo EXACTLY what data to use
    n_neighbors=30,
    basis=None,        # <--- prevents PCA
    layer=None,
)

# -------------------------------------------------
# RUN DYNAMO VELOCITY PROJECTION
# -------------------------------------------------
dyn.tl.cell_velocities(
    adata,
    ekey="X_raw",
    vkey="V_raw",
    X=X,
    V=V,
    X_embedding=adata.obsm["X_umap"],   # <- REQUIRED
    basis="umap",
    transition_genes=list(adata.var_names),
    method="pearson",
    enforce=True
)

V_emb = adata.obsm["velocity_umap"]

print("Velocity projection complete.")

# -------------------------------------------------
# MAKE STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata, basis="umap", color="color", pointsize=0.5,
    ax=ax, density=0.19, arrowsize=3.0, linewidth=3,
    streamline_alpha=1.0, xy_grid_nums=[50, 50],
    show_legend=False, show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

plt.show()

In [ ]:
dataset_name = "pancreas"

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
paths = DATASETS[dataset_name]

X = np.load(paths["X"])
V = np.load(paths["V"])
C = np.load(paths["color"])          # <-- real pseudotime
X_emb = np.load(paths["embedding"])  # <-- provided embedding

# ---- Clean pseudotime: ensure float + remove NaNs ----
C = np.asarray(C, dtype=float).ravel()
C = np.nan_to_num(C, nan=0.0)

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}, V shape: {V.shape}, embedding: {X_emb.shape}")

# -------------------------------------------------
# BUILD ANNDATA
# -------------------------------------------------
adata = anndata.AnnData(X)
adata.var_names = [f"dim{j}" for j in range(X.shape[1])]
adata.obs["color"] = C
adata.layers["X_raw"] = X
adata.layers["V_raw"] = V
adata.obsm["X_umap"] = X_emb

# -------------------------------------------------
# BUILD NEIGHBOR GRAPH (Dynamo needs this)
# -------------------------------------------------
dyn.tl.neighbors(
    adata,
    X_data=X,          # explicit: no PCA, no magic
    n_neighbors=30,
    basis=None,
    layer=None,
)

# -------------------------------------------------
# RUN DYNAMO VELOCITY PROJECTION
# -------------------------------------------------
dyn.tl.cell_velocities(
    adata,
    ekey="X_raw",
    vkey="V_raw",
    X=X,
    V=V,
    X_embedding=adata.obsm["X_umap"],
    basis="umap",
    transition_genes=list(adata.var_names),
    method="pearson",
    enforce=True,
)

V_emb = adata.obsm["velocity_umap"]
print("Velocity projection complete.")

# -------------------------------------------------
# STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,
    basis="umap",
    color="color",          # <-- true pseudotime
    pointsize=0.4,
    ax=ax,
    density=0.4,
    arrowsize=2.0,
    linewidth=1.5,
    streamline_alpha=1.0,
    xy_grid_nums=[50, 50],
    show_legend=False,
    show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

plt.show()

In [ ]:
import numpy as np
import anndata
import matplotlib.pyplot as plt
import dynamo as dyn

dataset_name = "dentate_gyrus"

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
paths = DATASETS[dataset_name]

X = np.load(paths["X"])
V = np.load(paths["V"])
C_raw = np.load(paths["color"])          # pseudotime
X_emb = np.load(paths["embedding"])      # fixed embedding

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}, V shape: {V.shape}, embedding: {X_emb.shape}")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# -------------------------------------------------
# FORCE COLOR INTO [0, 1] (paranoia mode)
# -------------------------------------------------
C = np.asarray(adata.obs["color"], dtype=float)
C = np.nan_to_num(C, nan=0.0)
C = (C - C.min()) / (C.max() - C.min() + 1e-12)
adata.obs["color"] = C

# -------------------------------------------------
# BLUE-SHIFT VIRIDIS: CUT OFF YELLOW
# -------------------------------------------------
# 0.0 → purple, ~0.7 → green, NO yellow
viridis_trunc = LinearSegmentedColormap.from_list(
    "viridis_trunc",
    plt.cm.viridis(np.linspace(0.0, 0.72, 256))
)


# -------------------------------------------------
# BUILD ANNDATA
# -------------------------------------------------
adata = anndata.AnnData(X)
adata.var_names = [f"dim{j}" for j in range(X.shape[1])]
adata.obs["color"] = C                  # ← continuous, numeric
adata.layers["X_raw"] = X
adata.layers["V_raw"] = V
adata.obsm["X_umap"] = X_emb

# -------------------------------------------------
# BUILD NEIGHBOR GRAPH (Dynamo requires this)
# -------------------------------------------------
dyn.tl.neighbors(
    adata,
    X_data=X,
    n_neighbors=30,
    basis=None,
    layer=None,
)

# -------------------------------------------------
# RUN DYNAMO VELOCITY PROJECTION
# -------------------------------------------------
dyn.tl.cell_velocities(
    adata,
    ekey="X_raw",
    vkey="V_raw",
    X=X,
    V=V,
    X_embedding=adata.obsm["X_umap"],
    basis="umap",
    transition_genes=list(adata.var_names),
    method="pearson",
    enforce=True,
)

print("Velocity projection complete.")
print("velocity_umap shape:", adata.obsm["velocity_umap"].shape)

# -------------------------------------------------
# STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,
    basis="umap",
    color="color",          # guaranteed [0, 1]
    cmap=viridis_trunc,
    pointsize=0.2,
    alpha=0.1,
    ax=ax,
    density=0.4,
    arrowsize=2.0,
    linewidth=1.5,
    streamline_alpha=1.0,
    xy_grid_nums=[50, 50],
    show_legend=False,
    show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

plt.show()

In [ ]:
dataset_name = "larry"

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
paths = DATASETS[dataset_name]

X = np.load(paths["X"])
V = np.load(paths["V"])
C = np.load(paths["color"])          # <-- real pseudotime
X_emb = np.load(paths["embedding"])  # <-- provided embedding

# ---- Clean pseudotime: ensure float + remove NaNs ----
C = np.asarray(C, dtype=float).ravel()
C = np.nan_to_num(C, nan=0.0)

print(f"\nLoaded dataset: {dataset_name}")
print(f"X shape: {X.shape}, V shape: {V.shape}, embedding: {X_emb.shape}")

# -------------------------------------------------
# BUILD ANNDATA
# -------------------------------------------------
adata = anndata.AnnData(X)
adata.var_names = [f"dim{j}" for j in range(X.shape[1])]
adata.obs["color"] = C
adata.layers["X_raw"] = X
adata.layers["V_raw"] = V
adata.obsm["X_umap"] = X_emb

# -------------------------------------------------
# BUILD NEIGHBOR GRAPH (Dynamo needs this)
# -------------------------------------------------
dyn.tl.neighbors(
    adata,
    X_data=X,          # explicit: no PCA, no magic
    n_neighbors=30,
    basis=None,
    layer=None,
)

# -------------------------------------------------
# RUN DYNAMO VELOCITY PROJECTION
# -------------------------------------------------
dyn.tl.cell_velocities(
    adata,
    ekey="X_raw",
    vkey="V_raw",
    X=X,
    V=V,
    X_embedding=adata.obsm["X_umap"],
    basis="umap",
    transition_genes=list(adata.var_names),
    method="pearson",
    enforce=True,
)

V_emb = adata.obsm["velocity_umap"]
print("Velocity projection complete.")

# -------------------------------------------------
# STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,
    basis="umap",
    color="color",          # <-- true pseudotime
    pointsize=0.2,
    alpha=0.1,
    ax=ax,
    density=0.4,
    arrowsize=2.0,
    linewidth=1.5,
    streamline_alpha=1.0,
    xy_grid_nums=[50, 50],
    show_legend=False,
    show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

plt.show()

In [ ]:
# -------------------------------------------------
# STREAMLINE PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

dyn.pl.streamline_plot(
    adata,
    basis="umap",
    color="color",          # <-- true pseudotime
    pointsize=0.1,
    alpha=0.05,
    ax=ax,
    density=0.4,
    arrowsize=2.0,
    linewidth=1.5,
    streamline_alpha=1.0,
    xy_grid_nums=[50, 50],
    show_legend=False,
    show_arrowed_spines=False,
    save_show_or_return="return",
)

ax.set_title(dataset_name)
ax.set_aspect("equal")
ax.axis("off")

plt.show()